# Esercitazione software 2

## Filtraggio di segnali musicali campionati 

Autori:

- Fissolo Emanuele - *s323585*
- Flora Alessandro - *s321504*
- Giraudo Giacomo - *s321784*
- Intagliata Francesco - *s325961*

### 1) Operazioni preliminari
In questa cella vengono eseguite le operazioni preliminari necessarie al funzionamento del codice, quali la lettura della traccia audio, la definizione della dimensione della finestra da analizzare etc...

In [ ]:
#in questo blocco eseguiamo tutte le operazione preliminari necessarie per il funzionamento dei
#blocchi di codice successivi

import numpy as np
from cmath import exp, pi
import matplotlib.pyplot as plt
import tes, soundfile, os
from tes import Tipo_filtro
from IPython.display import Audio, display

file_audio = str(os.path.join('input', 'CornfieldChase.oga'))
#file_audio = str(os.path.join('input', 'QueenBohemianRhapsody.oga'))

audio, fs = soundfile.read(file_audio)
audio = np.array(audio[:, 0], dtype=np.float32)

len_sec = len(audio)/fs

Df = 1/(len(audio)*1/fs)
x_axis_f = [x*Df for x in range(int(-len(audio)/2),int(len(audio)/2))]
x_axis_t = [x for x in range(0, len(audio))]

print(f"Estratti {len(audio)} campioni - campionati a {fs/1000:.1f}kHz - durata {len_sec} secondi - risoluzione in frequenza {Df:.2f}Hz")

#### Spettro del segnale originale

Procediamo a plottare lo spettro in frequenza della traccia audio originale scelta

In [ ]:
fft_py = np.abs(np.fft.fft(audio))

# calcoliamo il limite di frequenza per avere la banda al 99% del segnale
max_freq_banda = tes.get_limite_banda(tes.get_spettro(fft_py), Df, 99)
print(f"La frequenza corrispondente alla banda al 99% è {max_freq_banda}Hz")

# plottiamo la trasformata
plt.plot(x_axis_f, np.fft.fftshift(fft_py))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

#### Anteprima del file audio
La seguente cella consente di ottenere un'anteprima del file audio riproducendolo - la sua mancata riproduzione non inficia sul resto dell'elaborazione

In [ ]:
display(Audio(file_audio))

#### Prova applicazione filtro "Porta"

Generiamo un filtro "porta discreta nel tempo", che si comporta come un filtro passa-basso in frequenza. Vogliamo creare questo filtro in modo che abbia una frequenza di taglio a 3dB attorno alla frequenza 1KHz.

In [ ]:
len_filtro = tes.get_durata_per_taglio(1000, Tipo_filtro.PORTA, 0)
h_t = tes.get_porta_discreta(len_filtro, 0, fs, int(len_filtro*fs))
h_t = h_t / sum(h_t)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py_filtrato)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

Proviamo ad ascoltare il nuovo segnale audio

In [ ]:
file_out1 = str(os.path.join('output', 'CornfieldChaseFilter1.wav'))
soundfile.write(file=file_out1, data=audio_filtrato, samplerate=fs)

display(Audio(file_out1))

Come possiamo vedere dal grafico dello spettro in frequenza della traccia audio filtrata e come possiamo anche sentire ascoltando la traccia filtrata stessa, il fitro definito come una porta nel tempo ha uno scarso effetto in frequenza. Questo accade poichè la porta nel tempo si comporta come una sinc in frequenza, che decresce in maniera abbastanza lenta e successivamente oscilla andando a zero sono in alcune frequenze. 
Inoltre impostando la frequenza di taglio a 1KHz si vanno a includere nel risultato quasi tutte le frequenze della traccia originale (che cadono attorno a 600Hz e ai 300Hz)

#### Prova applicazione filtro "Coseno rialzato"

Generiamo un filtro "coseno rialzato nel tempo" con beta = 0.5, che si comporta sempre come un filtro passa-basso

In [ ]:
len_filtro = len_filtro = tes.get_durata_per_taglio(100, Tipo_filtro.COS_RIALZATO, 0.5)
h_t = tes.get_coseno_rialzato(len_filtro, len_filtro/2, fs, int(len_filtro*fs), 0.5)
h_t = h_t / sum(h_t)

audio_filtrato = np.convolve(audio, h_t, mode = "same")
fft_py_filtrato = np.abs(np.fft.fft(audio_filtrato))

plt.plot(x_axis_f, np.fft.fftshift(fft_py_filtrato))
plt.grid(True)
plt.axis([-max_freq_banda*1.25, max_freq_banda*1.25, 0, 1.25 * np.max(fft_py_filtrato)])
plt.xlabel("Frequency [Hz]")
plt.ylabel("Amplitude")
plt.show()

Proviamo ad ascoltare il nuovo segnale audio

In [ ]:
file_out2 = str(os.path.join('output', 'CornfieldChaseFilter2.wav'))
soundfile.write(file=file_out2, data=audio_filtrato, samplerate=fs)

display(Audio(file_out2))

In questo secondo caso possiamo invece notare un effetto molto più marcato del filtro sulla traccia audio. Impostando una frequenza di taglio a 100Hz (per escludere una buona parte delle frequenze più alte della nostra traccia audio) vediamo un notevole abbassamento nell'ampiezza di queste frequenze e ascoltando la traccia audio filtrata possiamo sentire un suono molto più ovattato, con i bassi estremamente più preponderanti rispetto alle frequenze più alte, che in certi casi risultano quasi completamente assenti